In [12]:
import sys

sys.path.append('/home/mdafifal.mamun/research/S3M')

In [13]:
import json
import sys
from sparse_recommendation import RecommendationFileSparse


def convert_sparse_to_list(sparse_filepath):
    """
    Convert a .sparse file to python list.
    
    Args:
        sparse_filepath: Path to the input .sparse file
    """
    # Read the sparse file
    sparse_file = RecommendationFileSparse(None, sparse_filepath, -1, only_read=True)
    
    results = []
    for query_id, candidates, scores in sparse_file.read_file():
        # Convert to the format expected by RecommendationFileJson
        # Format: (report_id, candidates, scores)
        # Convert numpy types to Python native types for JSON serialization
        candidates_list = [int(c) for c in candidates]
        scores_list = [float(s) for s in scores]
        results.append((int(query_id), candidates_list, scores_list))
    
    return results

In [14]:
other_method_predictions = "/work/disa_lab/afif/projects/TraceSim_EMSE/scripts/results/rebucket_netbeans_dedupt_results_4.sparse"
dedupt_predictions = "/home/mdafifal.mamun/research/S3M/netbeans_dedupt_predictions.json"

In [15]:
with open(dedupt_predictions, 'r') as f:
    dedupt_results = json.load(f)

In [16]:
dedupt_processed_results = []

for _, result in dedupt_results.items():
    bug_id = result["bug_id"]
    actual_duplicate_id = result["actual_duplicate_id"]
    preds = [int(pred) for pred in result["predictions"].keys()]

    dedupt_processed_results.append((bug_id, actual_duplicate_id, preds))
    

In [ ]:
other_method_results = convert_sparse_to_list(other_method_predictions)

In [18]:
other_method_processed_results = []

for bug_id, candidates, scores in other_method_results:
    other_method_processed_results.append((bug_id, candidates[:25]))

In [19]:
# Find common bug IDs between the two methods
common_bug_ids = set([result[0] for result in dedupt_processed_results]) & set([result[0] for result in other_method_processed_results])

# Keep only the results for the common bug IDs
dedupt_common_results = [result for result in dedupt_processed_results if result[0] in common_bug_ids]
other_method_common_results = [result for result in other_method_processed_results if result[0] in common_bug_ids]

In [20]:
len(dedupt_common_results)

1881

In [28]:
# Create a combined dataframe from the common results
import pandas as pd

combined_results = []

for dedupt_result in dedupt_common_results:
    bug_id, actual_duplicate_id, dedupt_preds = dedupt_result
    
    # Find the corresponding other method result
    other_method_result = next((result for result in other_method_common_results if result[0] == bug_id), None)
    
    if other_method_result is not None:
        _, other_method_preds = other_method_result
        combined_results.append({
            "bug_id": bug_id,
            "actual_duplicate_id": actual_duplicate_id,
            "dedupt_preds": dedupt_preds,
            "other_method_preds": other_method_preds
        })
combined_df = pd.DataFrame(combined_results)
combined_df.head(10)

,bug_id,actual_duplicate_id,dedupt_preds,other_method_preds
0,189530,185074,"[185074, 186242, 189385, 186421, 184026, 18453...","[163978, 158146, 188798, 176918, 157673, 15734..."
1,189539,186523,"[186523, 186991, 186051, 189116, 184053, 17716...","[189441, 186523, 188108, 169378, 158498, 18699..."
2,189551,167927,"[185937, 155207, 187074, 173803, 186946, 17326...","[185937, 175988, 167339, 155207, 155037, 15474..."
3,189573,167927,"[167927, 172850, 180395, 175603, 179025, 18642...","[181133, 159393, 163631, 167927, 163933, 17191..."
4,189588,187950,"[187950, 143495, 176700, 177033, 183067, 18070...","[187950, 166511, 186110, 143496, 143495, 17749..."
5,189608,189214,"[182504, 184570, 183130, 183160, 182677, 18921...","[183515, 184520, 183582, 189214, 188794, 18940..."
6,189617,187105,"[177812, 187105, 183749, 178580, 179853, 17405...","[188080, 187105, 183749, 181032, 177812, 14792..."
7,189618,187105,"[187105, 177812, 183749, 178580, 179853, 17858...","[183749, 187105, 188080, 181032, 177812, 14792..."
8,189636,179166,"[179166, 177024, 185740, 184633, 188513, 18751...","[179166, 151015, 180161, 189635, 189633, 18962..."
9,189641,189640,"[169336, 172516, 174003, 177593, 156876, 18483...","[183249, 179353, 174352, 173535, 170174, 16937..."


In [30]:
import numpy as np
from scipy.stats import wilcoxon

def reciprocal_rank(actual, preds):
    if actual in preds:
        return 1.0 / (preds.index(actual) + 1)
    return 0.0

rr_dedupt = []
rr_other = []

for index, row in combined_df.iterrows():
    rr_dedupt.append(reciprocal_rank(row["actual_duplicate_id"], row["dedupt_preds"]))
    rr_other.append(reciprocal_rank(row["actual_duplicate_id"], row["other_method_preds"]))

# Wilcoxon signed-rank test
stat, p_value = wilcoxon(rr_dedupt, rr_other)

print("p-value:", p_value)


p-value: 5.88054047474648e-112


In [21]:
import numpy as np
from scipy import stats
from scipy.stats import wilcoxon, ttest_rel
from statsmodels.stats.contingency_tables import mcnemar

def calculate_metrics(results_list, actual_id_map, k_values=[1, 5, 10]):
    """
    Calculate various ranking metrics.
    
    Args:
        results_list: List of (bug_id, predictions) tuples
        actual_id_map: Dictionary mapping bug_id to actual_duplicate_id
        k_values: List of k values for Accuracy@k
    
    Returns:
        Dictionary with metric names and values
    """
    metrics = {f'acc@{k}': 0 for k in k_values}
    metrics['mrr'] = 0  # Mean Reciprocal Rank
    
    reciprocal_ranks = []
    success_at_k = {k: [] for k in k_values}
    
    for bug_id, preds in results_list:
        actual_id = actual_id_map[bug_id]
        
        # Find rank of actual duplicate
        if actual_id in preds:
            rank = preds.index(actual_id) + 1  # 1-indexed
            reciprocal_ranks.append(1.0 / rank)
            
            # Check success at each k
            for k in k_values:
                success = 1 if rank <= k else 0
                success_at_k[k].append(success)
                metrics[f'acc@{k}'] += success
        else:
            reciprocal_ranks.append(0)
            for k in k_values:
                success_at_k[k].append(0)
    
    # Calculate final metrics
    n = len(results_list)
    for k in k_values:
        metrics[f'acc@{k}'] = metrics[f'acc@{k}'] / n
    
    metrics['mrr'] = sum(reciprocal_ranks) / n
    metrics['success_lists'] = success_at_k
    
    return metrics


def perform_statistical_tests(dedupt_success, other_method_success, metric_name):
    """
    Perform statistical significance tests.
    
    Args:
        dedupt_success: Binary list of successes for dedupt method
        other_method_success: Binary list of successes for other method
        metric_name: Name of the metric being tested
    
    Returns:
        Dictionary with test results
    """
    results = {'metric': metric_name}
    
    # McNemar's test (for paired binary outcomes)
    # Create contingency table
    both_correct = sum([1 for d, o in zip(dedupt_success, other_method_success) if d == 1 and o == 1])
    dedupt_only = sum([1 for d, o in zip(dedupt_success, other_method_success) if d == 1 and o == 0])
    other_only = sum([1 for d, o in zip(dedupt_success, other_method_success) if d == 0 and o == 1])
    both_wrong = sum([1 for d, o in zip(dedupt_success, other_method_success) if d == 0 and o == 0])
    
    contingency_table = [[both_correct, dedupt_only],
                         [other_only, both_wrong]]
    
    try:
        mcnemar_result = mcnemar(contingency_table, exact=True)
        results['mcnemar_statistic'] = mcnemar_result.statistic
        results['mcnemar_pvalue'] = mcnemar_result.pvalue
    except Exception as e:
        results['mcnemar_error'] = str(e)
    
    # Wilcoxon signed-rank test (non-parametric paired test)
    try:
        wilcoxon_result = wilcoxon(dedupt_success, other_method_success, alternative='two-sided')
        results['wilcoxon_statistic'] = wilcoxon_result.statistic
        results['wilcoxon_pvalue'] = wilcoxon_result.pvalue
    except Exception as e:
        results['wilcoxon_error'] = str(e)
    
    # Effect size information
    results['dedupt_wins'] = dedupt_only
    results['other_wins'] = other_only
    results['ties'] = both_correct + both_wrong
    results['improvement'] = dedupt_only - other_only
    
    return results

In [22]:
# Build mapping from bug_id to actual_duplicate_id
actual_id_map = {}
for bug_id, actual_duplicate_id, preds in dedupt_common_results:
    actual_id_map[bug_id] = actual_duplicate_id

print(f"Created mapping for {len(actual_id_map)} bug reports")

Created mapping for 1881 bug reports


In [23]:
# Prepare data for both methods
dedupt_formatted = [(bug_id, preds) for bug_id, actual_id, preds in dedupt_common_results]
other_method_formatted = other_method_common_results

# Calculate metrics for both methods
print("="*60)
print("EVALUATION METRICS")
print("="*60)

dedupt_metrics = calculate_metrics(dedupt_formatted, actual_id_map, k_values=[1, 5, 10])
other_method_metrics = calculate_metrics(other_method_formatted, actual_id_map, k_values=[1, 5, 10])

# Display results
print("\nDEDUPT Method:")
print(f"  Accuracy@1:  {dedupt_metrics['acc@1']:.4f}")
print(f"  Accuracy@5:  {dedupt_metrics['acc@5']:.4f}")
print(f"  Accuracy@10: {dedupt_metrics['acc@10']:.4f}")
print(f"  MRR:         {dedupt_metrics['mrr']:.4f}")

print("\nOther Method:")
print(f"  Accuracy@1:  {other_method_metrics['acc@1']:.4f}")
print(f"  Accuracy@5:  {other_method_metrics['acc@5']:.4f}")
print(f"  Accuracy@10: {other_method_metrics['acc@10']:.4f}")
print(f"  MRR:         {other_method_metrics['mrr']:.4f}")

print("\nImprovement (DEDUPT - Other):")
print(f"  Δ Accuracy@1:  {(dedupt_metrics['acc@1'] - other_method_metrics['acc@1']):.4f} ({((dedupt_metrics['acc@1'] - other_method_metrics['acc@1']) / other_method_metrics['acc@1'] * 100):.2f}%)")
print(f"  Δ Accuracy@5:  {(dedupt_metrics['acc@5'] - other_method_metrics['acc@5']):.4f} ({((dedupt_metrics['acc@5'] - other_method_metrics['acc@5']) / other_method_metrics['acc@5'] * 100):.2f}%)")
print(f"  Δ Accuracy@10: {(dedupt_metrics['acc@10'] - other_method_metrics['acc@10']):.4f} ({((dedupt_metrics['acc@10'] - other_method_metrics['acc@10']) / other_method_metrics['acc@10'] * 100):.2f}%)")
print(f"  Δ MRR:         {(dedupt_metrics['mrr'] - other_method_metrics['mrr']):.4f} ({((dedupt_metrics['mrr'] - other_method_metrics['mrr']) / other_method_metrics['mrr'] * 100):.2f}%)")

EVALUATION METRICS

DEDUPT Method:
  Accuracy@1:  0.6443
  Accuracy@5:  0.8278
  Accuracy@10: 0.8655
  MRR:         0.7265

Other Method:
  Accuracy@1:  0.3344
  Accuracy@5:  0.5667
  Accuracy@10: 0.6417
  MRR:         0.4375

Improvement (DEDUPT - Other):
  Δ Accuracy@1:  0.3099 (92.69%)
  Δ Accuracy@5:  0.2610 (46.06%)
  Δ Accuracy@10: 0.2238 (34.88%)
  Δ MRR:         0.2890 (66.05%)


In [26]:
# Statistical Significance Tests
print("\n" + "="*60)
print("STATISTICAL SIGNIFICANCE TESTS")
print("="*60)

k_values = [1, 5, 10]
all_test_results = []

for k in k_values:
    print(f"\n--- Accuracy@{k} ---")
    
    dedupt_success = dedupt_metrics['success_lists'][k]
    other_success = other_method_metrics['success_lists'][k]
    
    test_results = perform_statistical_tests(dedupt_success, other_success, f'Accuracy@{k}')
    all_test_results.append(test_results)
    
    print(f"  DEDUPT wins: {test_results['dedupt_wins']}")
    print(f"  Other wins:  {test_results['other_wins']}")
    print(f"  Net improvement: {test_results['improvement']}")
    
    if 'mcnemar_pvalue' in test_results:
        print(f"\n  McNemar's test:")
        print(f"    Statistic: {test_results['mcnemar_statistic']}")
        print(f"    p-value: {test_results['mcnemar_pvalue']}")
        if test_results['mcnemar_pvalue'] < 0.001:
            print(f"    Result: *** Highly significant (p < 0.001)")
        elif test_results['mcnemar_pvalue'] < 0.01:
            print(f"    Result: ** Very significant (p < 0.01)")
        elif test_results['mcnemar_pvalue'] < 0.05:
            print(f"    Result: * Significant (p < 0.05)")
        else:
            print(f"    Result: Not significant (p >= 0.05)")
    
    if 'wilcoxon_pvalue' in test_results:
        print(f"\n  Wilcoxon signed-rank test:")
        print(f"    Statistic: {test_results['wilcoxon_statistic']}")
        print(f"    p-value: {test_results['wilcoxon_pvalue']}")
        if test_results['wilcoxon_pvalue'] < 0.001:
            print(f"    Result: *** Highly significant (p < 0.001)")
        elif test_results['wilcoxon_pvalue'] < 0.01:
            print(f"    Result: ** Very significant (p < 0.01)")
        elif test_results['wilcoxon_pvalue'] < 0.05:
            print(f"    Result: * Significant (p < 0.05)")
        else:
            print(f"    Result: Not significant (p >= 0.05)")

print("\n" + "="*60)
print(f"Total test cases: {len(dedupt_common_results)}")
print("="*60)


STATISTICAL SIGNIFICANCE TESTS

--- Accuracy@1 ---
  DEDUPT wins: 680
  Other wins:  97
  Net improvement: 583

  McNemar's test:
    Statistic: 97.0
    p-value: 1.3728740585261796e-108
    Result: *** Highly significant (p < 0.001)

  Wilcoxon signed-rank test:
    Statistic: 37733.0
    p-value: 3.909589250073686e-97
    Result: *** Highly significant (p < 0.001)

--- Accuracy@5 ---
  DEDUPT wins: 572
  Other wins:  81
  Net improvement: 491

  McNemar's test:
    Statistic: 81.0
    p-value: 6.164979410491018e-92
    Result: *** Highly significant (p < 0.001)

  Wilcoxon signed-rank test:
    Statistic: 26487.0
    p-value: 2.809183232668458e-82
    Result: *** Highly significant (p < 0.001)

--- Accuracy@10 ---
  DEDUPT wins: 492
  Other wins:  71
  Net improvement: 421

  McNemar's test:
    Statistic: 71.0
    p-value: 1.7481034956231055e-78
    Result: *** Highly significant (p < 0.001)

  Wilcoxon signed-rank test:
    Statistic: 20022.0
    p-value: 1.951079588000316e-70
   

## Statistical Significance Tests

We'll use two complementary tests to determine if the performance difference is statistically significant:

1. **McNemar's Test**: Tests if there's a significant difference between two paired binary outcomes (success/failure). Particularly useful for comparing two methods on the same dataset.
   - Null hypothesis: Both methods have the same error rate
   - p < 0.05: Significant difference between methods

2. **Wilcoxon Signed-Rank Test**: Non-parametric test for paired samples that doesn't assume normal distribution.
   - Tests if the median difference between pairs is zero
   - p < 0.05: Significant difference between methods

**Significance Levels:**
- p < 0.001: *** (highly significant)
- p < 0.01: ** (very significant)  
- p < 0.05: * (significant)
- p >= 0.05: Not significant

**Interpretation:**
- If p-value < 0.05, the difference is statistically significant
- "Net improvement" shows which method performs better (positive = DEDUPT better, negative = Other method better)
- Both tests should agree if the difference is robust